In [ ]:
RUNNER = "quang"                 # hoặc "trong"
RUNNERS = ("quang", "trong")
RUN_SCOPE = "PREFLIGHT"          # PREFLIGHT hoặc SHARD
CONFIRM_SHARD = ""               # SHARD phải đúng RUNNER
SESSION_MAX_MINUTES = 600
BGE_BATCH_SIZE = 16

# 28 — B1 BGE-M3 producer (GPU, resumable, research-only)

Notebook này nhúng **chỉ** caption `qwen_captions` đã được notebook 26 xác minh
cho đúng 5.301 candidate B1. Nó không đọc ảnh/video, không chạy lại Qwen/OCR,
không build Faiss/BM25 và không sửa BTC base.

- Mặc định `PREFLIGHT`: xác minh lineage + Qwen runner manifest, load BGE một
  lần, encode lặp lại ít nhất hai caption và **không ghi** output research chính.
- `SHARD`: chỉ mở khi `CONFIRM_SHARD == RUNNER` và Qwen manifest của chính
  runner đó là `COMPLETE`.
- Output research: `parts/dense_vectors/{video_id}.npz`, marker theo video và
  `runner-manifests/{runner}.json` trên volume `keyframe-augmentation-research`.

Chạy trong một Kaggle session mới với GPU T4 (hoặc GPU tương thích torch hiện
tại), Internet bật và ba secrets `GITHUB_TOKEN`, `MODAL_TOKEN_ID`,
`MODAL_TOKEN_SECRET`. Không chạy chung environment Qwen/vLLM.

## 1 — Constants, contracts và helper có thể unit-test

In [ ]:
from __future__ import annotations

import json
import math
import shutil
from collections import defaultdict
from pathlib import Path
from typing import Any, Callable

EXPECTED_PARITY_FREEZE_SHA256 = "3b9993755bb19000e80a9c7bd9ccdde1e9a4648dd1950167d0759f9a14227180"
EXPECTED_NB24_COMPLETE_SHA256 = "39950f697a1bfa9d6e6c9b78e220b8b62da48a99d8465b6f9b46a9c6b16a5c10"
EXPECTED_B1_COUNT = 5301
CHECKPOINT = "BAAI/bge-m3"
VECTOR_DIM = 1024
VECTOR_DTYPE = "float32"
NORM_ATOL = 1e-3
PIPELINE = "bge_m3"
QWEN_PIPELINE = "qwen25"
QWEN_ROOT = "/research/keyframe-augmentation/transnetv2-btc-b1-v1/gate1-b1-qwen25-v1"
OUTPUT_ROOT = "/research/keyframe-augmentation/transnetv2-btc-b1-v1/gate1-b1-bge-m3-v1"
MODEL_CACHE_KEY = f"bge-m3 {CHECKPOINT}"


def _payload_hash_matches(payload: dict[str, Any], field: str, hash_json: Callable[[Any], str]) -> bool:
    unsigned = dict(payload)
    claimed = unsigned.pop(field, None)
    return isinstance(claimed, str) and claimed == hash_json(unsigned)


def validate_run_lock(runner: str, runners: tuple[str, ...], scope: str, confirm: str) -> None:
    if runner not in runners:
        raise RuntimeError(f"Unknown runner: {runner!r}")
    if scope not in {"PREFLIGHT", "SHARD"}:
        raise RuntimeError("RUN_SCOPE must be PREFLIGHT or SHARD")
    if scope == "SHARD" and confirm != runner:
        raise RuntimeError("SHARD is locked: set CONFIRM_SHARD exactly to RUNNER")


def split_b1_rows(
    rows: list[dict[str, Any]],
    runners: tuple[str, ...],
    *,
    expected_count: int = EXPECTED_B1_COUNT,
) -> tuple[dict[str, dict[str, list[dict[str, Any]]]], dict[str, int]]:
    b1_rows = [dict(row) for row in rows if "B1" in row.get("arms", [])]
    identities = [str(row["keyframe_id"]) for row in b1_rows]
    if len(b1_rows) != expected_count or len(set(identities)) != expected_count:
        raise RuntimeError("B1 identity/count drift")
    by_video: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for row in b1_rows:
        by_video[str(row["video_id"])].append(row)
    loads = {runner: 0 for runner in runners}
    owner: dict[str, str] = {}
    for video_id, video_rows in sorted(by_video.items(), key=lambda item: (-len(item[1]), item[0])):
        runner = min(runners, key=lambda value: (loads[value], value))
        owner[video_id] = runner
        loads[runner] += len(video_rows)
    partitions: dict[str, dict[str, list[dict[str, Any]]]] = {runner: {} for runner in runners}
    for video_id, video_rows in by_video.items():
        runner = owner[video_id]
        partitions[runner][video_id] = sorted(video_rows, key=lambda row: int(row["frame_idx"]))
    return partitions, loads


def validate_qwen_runner_manifest(
    manifest: dict[str, Any],
    *,
    runner: str,
    assigned_videos: int,
    assigned_frames: int,
    expected_nb24_sha: str,
    hash_json: Callable[[Any], str],
) -> None:
    if not _payload_hash_matches(manifest, "manifest_payload_hash", hash_json):
        raise RuntimeError("Qwen runner manifest payload hash mismatch")
    checks = {
        "status": manifest.get("status") == "COMPLETE",
        "pipeline": manifest.get("pipeline") == QWEN_PIPELINE,
        "runner": manifest.get("runner") == runner,
        "assigned_videos": manifest.get("assigned_videos") == assigned_videos,
        "verified_videos": manifest.get("verified_videos") == assigned_videos,
        "assigned_frames": manifest.get("assigned_frames") == assigned_frames,
        "verified_frames": manifest.get("verified_frames") == assigned_frames,
        "nb24_lineage": manifest.get("nb24_complete_sha256") == expected_nb24_sha,
    }
    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise RuntimeError(f"Qwen runner manifest is not COMPLETE/compatible: {failed}")


def validate_qwen_marker(
    marker: dict[str, Any],
    *,
    runner: str,
    video_id: str,
    expected_ids_hash: str,
    expected_frames: int,
    expected_nb24_sha: str,
    hash_json: Callable[[Any], str],
) -> tuple[str, dict[str, Any]]:
    if not _payload_hash_matches(marker, "marker_payload_hash", hash_json):
        raise RuntimeError(f"{video_id}: Qwen marker payload hash mismatch")
    checks = {
        "status": marker.get("status") == "COMPLETE",
        "pipeline": marker.get("pipeline") == QWEN_PIPELINE,
        "runner": marker.get("runner") == runner,
        "video_id": marker.get("video_id") == video_id,
        "frame_count": marker.get("frame_count") == expected_frames,
        "identity": marker.get("keyframe_ids_sha256") == expected_ids_hash,
        "nb24_lineage": marker.get("nb24_complete_sha256") == expected_nb24_sha,
    }
    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise RuntimeError(f"{video_id}: Qwen marker contract mismatch: {failed}")
    artifacts = marker.get("artifacts")
    if not isinstance(artifacts, dict) or set(artifacts) != {"qwen_captions", "qwen_ocr"}:
        raise RuntimeError(f"{video_id}: Qwen artifact set must be captions + OCR")
    caption_artifact = artifacts["qwen_captions"]
    expected_remote = f"{QWEN_ROOT}/parts/qwen_captions/{video_id}.json"
    if caption_artifact.get("remote") != expected_remote:
        raise RuntimeError(f"{video_id}: Qwen caption path drift")
    expected_ocr_remote = f"{QWEN_ROOT}/parts/qwen_ocr/{video_id}.json"
    if artifacts["qwen_ocr"].get("remote") != expected_ocr_remote:
        raise RuntimeError(f"{video_id}: Qwen OCR path drift")
    if not isinstance(caption_artifact.get("sha256"), str):
        raise RuntimeError(f"{video_id}: Qwen caption SHA is missing")
    return str(marker["marker_payload_hash"]), caption_artifact


def load_and_validate_qwen_video(
    transport: Any,
    workdir: Path,
    *,
    runner: str,
    video_id: str,
    expected_ids: list[str],
    expected_nb24_sha: str,
    hash_json: Callable[[Any], str],
    hash_file: Callable[[Path], str],
) -> tuple[dict[str, str], dict[str, Any]]:
    marker_remote = f"{QWEN_ROOT}/markers/{runner}/{video_id}.json"
    marker_local = workdir / "qwen-markers" / runner / f"{video_id}.json"
    transport.get(marker_remote, marker_local)
    marker = json.loads(marker_local.read_text(encoding="utf-8"))
    marker_hash, caption_artifact = validate_qwen_marker(
        marker,
        runner=runner,
        video_id=video_id,
        expected_ids_hash=hash_json(expected_ids),
        expected_frames=len(expected_ids),
        expected_nb24_sha=expected_nb24_sha,
        hash_json=hash_json,
    )
    for artifact in marker["artifacts"].values():
        if transport.sha256(str(artifact["remote"])) != artifact["sha256"]:
            raise RuntimeError(f"{video_id}: Qwen artifact remote SHA mismatch")
    caption_local = workdir / "qwen-captions" / f"{video_id}.json"
    transport.get(str(caption_artifact["remote"]), caption_local)
    if hash_file(caption_local) != caption_artifact["sha256"]:
        raise RuntimeError(f"{video_id}: downloaded Qwen caption SHA mismatch")
    captions_raw = json.loads(caption_local.read_text(encoding="utf-8"))
    if not isinstance(captions_raw, dict):
        raise RuntimeError(f"{video_id}: Qwen captions must be a JSON object")
    expected = set(expected_ids)
    captions: dict[str, str] = {}
    for key, value in captions_raw.items():
        if key not in expected:
            raise RuntimeError(f"{video_id}: caption key outside frozen B1 worklist: {key}")
        text = str(value).strip()
        if not text:
            raise RuntimeError(f"{video_id}: empty captions must not be embedded")
        captions[str(key)] = text
    if int(caption_artifact.get("rows", -1)) != len(captions):
        raise RuntimeError(f"{video_id}: Qwen caption row count mismatch")
    if not captions:
        raise RuntimeError(f"{video_id}: no non-empty Qwen caption to embed")
    provenance = {
        "qwen_marker_payload_hash": marker_hash,
        "qwen_caption_sha256": str(caption_artifact["sha256"]),
        "qwen_caption_remote": str(caption_artifact["remote"]),
    }
    return {key: captions[key] for key in sorted(captions)}, provenance


def inspect_bge_npz(
    path: Path,
    *,
    expected_ids: list[str],
    video_id: str,
    norm_atol: float = NORM_ATOL,
) -> dict[str, Any]:
    with np.load(path, allow_pickle=False) as payload:
        if set(payload.files) != {"ids", "vectors"}:
            raise RuntimeError(f"{video_id}: unexpected NPZ members")
        raw_ids = payload["ids"]
        vectors = payload["vectors"]
    if raw_ids.dtype.kind != "U":
        raise RuntimeError(f"{video_id}: ids must be fixed-width Unicode")
    ids = [str(value) for value in raw_ids.tolist()]
    if ids != expected_ids or len(ids) != len(set(ids)):
        raise RuntimeError(f"{video_id}: NPZ IDs/order mismatch")
    if vectors.dtype != np.float32:
        raise RuntimeError(f"{video_id}: vectors must be float32")
    if vectors.shape != (len(ids), VECTOR_DIM):
        raise RuntimeError(f"{video_id}: vector shape mismatch: {vectors.shape}")
    if not np.isfinite(vectors).all():
        raise RuntimeError(f"{video_id}: vectors contain non-finite values")
    norms = np.linalg.norm(vectors, axis=1)
    if len(norms) and not np.allclose(norms, 1.0, atol=norm_atol, rtol=0.0):
        raise RuntimeError(f"{video_id}: vectors are not L2-normalized")
    if any(not value.startswith(f"{video_id}#") for value in ids):
        raise RuntimeError(f"{video_id}: NPZ contains an ID from another video")
    return {
        "rows": len(ids),
        "dim": VECTOR_DIM,
        "dtype": VECTOR_DTYPE,
        "max_norm_error": float(np.max(np.abs(norms - 1.0))) if len(norms) else 0.0,
    }


def completed(
    transport: Any,
    workdir: Path,
    *,
    marker_remote: str,
    expected_marker_fields: dict[str, Any],
    vector_remote: str,
    expected_ids: list[str],
    video_id: str,
    hash_json: Callable[[Any], str],
    hash_file: Callable[[Path], str],
) -> dict[str, Any] | None:
    marker_local = workdir / "resume-markers" / f"{video_id}.json"
    vector_local = workdir / "resume-vectors" / f"{video_id}.npz"
    try:
        transport.get(marker_remote, marker_local)
        marker = json.loads(marker_local.read_text(encoding="utf-8"))
        if not _payload_hash_matches(marker, "marker_payload_hash", hash_json):
            return None
        if any(marker.get(key) != value for key, value in expected_marker_fields.items()):
            return None
        source_commit = marker.get("repo_commit")
        if not isinstance(source_commit, str) or len(source_commit) != 40:
            return None
        artifact = marker.get("artifact")
        if not isinstance(artifact, dict) or artifact.get("remote") != vector_remote:
            return None
        digest = artifact.get("sha256")
        if not isinstance(digest, str) or transport.sha256(vector_remote) != digest:
            return None
        transport.get(vector_remote, vector_local)
        if hash_file(vector_local) != digest:
            return None
        inspect_bge_npz(vector_local, expected_ids=expected_ids, video_id=video_id)
        return marker
    except (FileNotFoundError, json.JSONDecodeError, KeyError, TypeError, ValueError, RuntimeError):
        return None
    finally:
        marker_local.unlink(missing_ok=True)
        vector_local.unlink(missing_ok=True)


def build_runner_manifest(
    *,
    runner: str,
    assigned: dict[str, list[dict[str, Any]]],
    verified: dict[str, dict[str, Any]],
    failures: list[dict[str, str]],
    environment: dict[str, str],
    repo_commit: str,
    model_revision: str,
    model_fingerprint: str,
) -> dict[str, Any]:
    assigned_frames = sum(len(rows) for rows in assigned.values())
    verified_frames = sum(len(assigned[video_id]) for video_id in verified)
    caption_rows = sum(int(marker["caption_row_count"]) for marker in verified.values())
    vector_rows = sum(int(marker["vector_row_count"]) for marker in verified.values())
    return {
        "schema_version": 1,
        "status": "COMPLETE" if set(verified) == set(assigned) else "PARTIAL",
        "pipeline": PIPELINE,
        "runner": runner,
        "assigned_videos": len(assigned),
        "assigned_frames": assigned_frames,
        "verified_videos": len(verified),
        "verified_frames": verified_frames,
        "non_empty_caption_rows": caption_rows,
        "vector_rows": vector_rows,
        "caption_coverage": caption_rows / assigned_frames if assigned_frames else 0.0,
        "feature_coverage_status": (
            "FULL" if caption_rows == assigned_frames else "GAP_REPORTED_NO_ZERO_FILL"
        ),
        "failures_this_session": failures,
        "checkpoint": CHECKPOINT,
        "model_revision": model_revision,
        "model_snapshot_fingerprint": model_fingerprint,
        "environment": environment,
        "parity_freeze_sha256": EXPECTED_PARITY_FREEZE_SHA256,
        "nb24_complete_sha256": EXPECTED_NB24_COMPLETE_SHA256,
        "repo_commit": repo_commit,
        "heldout_accessed": False,
        "btc_mutated": False,
    }


def load_bge_model(model_factory: Callable[[], Any], cached_model_fn: Callable[[str, Callable[[], Any]], Any]) -> Any:
    return cached_model_fn(MODEL_CACHE_KEY, model_factory)

## 2 — Repo, isolated BGE environment, GPU và Modal

In [ ]:
import hashlib
import os
import subprocess
import sys
import time
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang/storage"
REPO = Path("/kaggle/working/multimodal-fusion-engine")
WORK = Path("/kaggle/working/b1-bge-m3-producer")
REMOTE = WORK / "remote"
OUTPUT = WORK / "output"
for directory in (WORK, REMOTE, OUTPUT):
    directory.mkdir(parents=True, exist_ok=True)


def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> str:
    result = subprocess.run(command, cwd=cwd, env=env, check=False, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {' '.join(command[:4])}\n{result.stderr[-1800:]}")
    return result.stdout.strip()


validate_run_lock(RUNNER, RUNNERS, RUN_SCOPE, CONFIRM_SHARD)
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")
askpass = WORK / "git-askpass.sh"
askpass.write_text(
    '#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n',
    encoding="utf-8",
)
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({"GITHUB_TOKEN": github_token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
try:
    if (REPO / ".git").is_dir():
        if run(["git", "status", "--porcelain"], cwd=REPO):
            raise RuntimeError("Kaggle checkout has local changes; restart the session")
        run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO, env=git_env)
        run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
    else:
        run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], env=git_env)
finally:
    askpass.unlink(missing_ok=True)

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "notebooks/requirements-kaggle-btc-bge.txt")])
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import torch
import yaml
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer

from mfe.offline.bundle import check_vectors
from mfe.offline.keyframe_augmentation.common import atomic_write_json, read_jsonl, sha256_file, sha256_json
from mfe.offline.stages import StageContext, cached_model, stage_n6_bge_m3, unload_models
from mfe.offline.transport import ModalVolumeTransport

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 28 requires a compatible GPU")
capability = torch.cuda.get_device_capability(0)
required_arch = f"sm_{capability[0]}{capability[1]}"
if capability[0] <= 6 or required_arch not in torch.cuda.get_arch_list():
    raise RuntimeError(f"Unsupported GPU/torch architecture: {capability}, {torch.cuda.get_arch_list()}")
try:
    version("vllm")
except PackageNotFoundError:
    pass
else:
    raise RuntimeError("Use a fresh BGE-only session; a Qwen/vLLM environment is not allowed")

repo_commit = run(["git", "rev-parse", "HEAD"], cwd=REPO)
cfg = yaml.safe_load((REPO / "config/keyframe_augmentation_kaggle.yaml").read_text(encoding="utf-8"))
if cfg["output"]["volume"] != "keyframe-augmentation-research":
    raise RuntimeError("Research volume drift")
research = ModalVolumeTransport(cfg["output"]["volume"], create_if_missing=False, version=2)
root = cfg["output"]["root"].rstrip("/")
nb23_root = f"{root}/gate1-feature-parity-worklist-v1"
nb24_root = f"{root}/gate1-siglip-b0-b4-screen-v1"
if QWEN_ROOT != f"{root}/gate1-b1-qwen25-v1" or OUTPUT_ROOT != f"{root}/gate1-b1-bge-m3-v1":
    raise RuntimeError("Pinned research output roots drifted")

PACKAGE_VERSIONS = {name: version(name) for name in ("torch", "sentence-transformers", "transformers", "numpy")}
print({"repo_commit": repo_commit, "gpu": torch.cuda.get_device_name(0), "scope": RUN_SCOPE, "runner": RUNNER})
print({"resolved_packages": PACKAGE_VERSIONS})
print("Sau PREFLIGHT đầu tiên, pin bốn phiên bản trên trong requirements-kaggle-btc-bge.txt rồi commit cho cả hai runner.")

## 3 — Fail-closed lineage, B1 split và Qwen runner dependency

In [ ]:
nb23_complete_path = REMOTE / "nb23-COMPLETE.json"
parity_freeze_path = REMOTE / "parity-freeze.json"
worklist_path = REMOTE / "feature-parity-worklist.jsonl"
nb24_complete_path = REMOTE / "nb24-COMPLETE.json"
research.get(f"{nb23_root}/COMPLETE.json", nb23_complete_path)
research.get(f"{nb23_root}/parity-freeze.json", parity_freeze_path)
research.get(f"{nb23_root}/feature-parity-worklist.jsonl", worklist_path)
research.get(f"{nb24_root}/COMPLETE.json", nb24_complete_path)

if sha256_file(parity_freeze_path) != EXPECTED_PARITY_FREEZE_SHA256:
    raise RuntimeError("Notebook 23 parity freeze SHA drift")
if sha256_file(nb24_complete_path) != EXPECTED_NB24_COMPLETE_SHA256:
    raise RuntimeError("Notebook 24 COMPLETE SHA drift")

nb23_complete = json.loads(nb23_complete_path.read_text(encoding="utf-8"))
parity_freeze = json.loads(parity_freeze_path.read_text(encoding="utf-8"))
nb24_complete = json.loads(nb24_complete_path.read_text(encoding="utf-8"))
if not _payload_hash_matches(nb23_complete, "manifest_payload_hash", sha256_json):
    raise RuntimeError("Notebook 23 COMPLETE payload hash mismatch")
if not _payload_hash_matches(parity_freeze, "freeze_payload_hash", sha256_json):
    raise RuntimeError("Notebook 23 parity freeze payload hash mismatch")
if not _payload_hash_matches(nb24_complete, "manifest_payload_hash", sha256_json):
    raise RuntimeError("Notebook 24 COMPLETE payload hash mismatch")
if nb23_complete.get("status") != "COMPLETE" or parity_freeze.get("status") != "READY_FOR_FEATURE_PARITY_PRODUCERS":
    raise RuntimeError("Notebook 23 lineage is not ready")
if nb23_complete.get("parity_freeze_sha256") != EXPECTED_PARITY_FREEZE_SHA256:
    raise RuntimeError("Notebook 23 COMPLETE/parity lineage mismatch")
if nb24_complete.get("provisional_winner") != "B1":
    raise RuntimeError("Notebook 24 provisional arm is not B1")
if sha256_file(worklist_path) != parity_freeze["files"]["feature-parity-worklist.jsonl"]:
    raise RuntimeError("Notebook 23 worklist SHA mismatch")

all_rows = list(read_jsonl(worklist_path))
partitions, runner_loads = split_b1_rows(all_rows, RUNNERS)
assigned = partitions[RUNNER]
assigned_frames = sum(len(rows) for rows in assigned.values())

qwen_manifest_remote = f"{QWEN_ROOT}/runner-manifests/{RUNNER}.json"
qwen_manifest_local = REMOTE / f"qwen-runner-manifest-{RUNNER}.json"
research.get(qwen_manifest_remote, qwen_manifest_local)
qwen_manifest = json.loads(qwen_manifest_local.read_text(encoding="utf-8"))
validate_qwen_runner_manifest(
    qwen_manifest,
    runner=RUNNER,
    assigned_videos=len(assigned),
    assigned_frames=assigned_frames,
    expected_nb24_sha=EXPECTED_NB24_COMPLETE_SHA256,
    hash_json=sha256_json,
)
print({
    "lineage": "PASS",
    "qwen_dependency": "COMPLETE",
    "b1_frames": EXPECTED_B1_COUNT,
    "runner_loads": runner_loads,
    "assigned_videos": len(assigned),
    "assigned_frames": assigned_frames,
})

## 4 — Resolve model provenance before loading BGE

In [ ]:
def snapshot_fingerprint(snapshot_dir: Path) -> str:
    entries = []
    for path in sorted(snapshot_dir.rglob("*")):
        if not path.is_file():
            continue
        resolved = path.resolve()
        entries.append({
            "path": path.relative_to(snapshot_dir).as_posix(),
            "bytes": path.stat().st_size,
            "blob": resolved.name,
        })
    if not entries:
        raise RuntimeError("BGE model snapshot is empty")
    return sha256_json(entries)


snapshot_dir = Path(snapshot_download(CHECKPOINT))
MODEL_REVISION = snapshot_dir.name
MODEL_SNAPSHOT_FINGERPRINT = snapshot_fingerprint(snapshot_dir)
print({
    "checkpoint": CHECKPOINT,
    "resolved_revision": MODEL_REVISION,
    "model_snapshot_fingerprint": MODEL_SNAPSHOT_FINGERPRINT,
    "model_loaded": False,
})

## 5 — PREFLIGHT (không ghi output research chính)

In [ ]:
if RUN_SCOPE == "PREFLIGHT":
    selected: list[tuple[str, str]] = []
    selected_source: list[str] = []
    for video_id in sorted(assigned):
        rows = assigned[video_id]
        expected_ids = [str(row["keyframe_id"]) for row in rows]
        captions, _provenance = load_and_validate_qwen_video(
            research,
            WORK,
            runner=RUNNER,
            video_id=video_id,
            expected_ids=expected_ids,
            expected_nb24_sha=EXPECTED_NB24_COMPLETE_SHA256,
            hash_json=sha256_json,
            hash_file=sha256_file,
        )
        for key, text in captions.items():
            selected.append((key, text))
            selected_source.append(video_id)
            if len(selected) == 2:
                break
        if len(selected) == 2:
            break
    if len(selected) < 2:
        raise RuntimeError("PREFLIGHT requires at least two verified Qwen captions")

    model = load_bge_model(
        lambda: SentenceTransformer(str(snapshot_dir)),
        cached_model,
    )
    texts = [text for _key, text in selected]
    first = np.asarray(
        model.encode(texts, normalize_embeddings=True, batch_size=min(BGE_BATCH_SIZE, 2), convert_to_numpy=True),
        dtype=np.float32,
    )
    second = np.asarray(
        model.encode(texts, normalize_embeddings=True, batch_size=min(BGE_BATCH_SIZE, 2), convert_to_numpy=True),
        dtype=np.float32,
    )
    if first.shape != (2, VECTOR_DIM) or second.shape != first.shape:
        raise RuntimeError(f"PREFLIGHT vector shape mismatch: {first.shape}, {second.shape}")
    if not np.isfinite(first).all() or not np.isfinite(second).all():
        raise RuntimeError("PREFLIGHT produced non-finite vectors")
    norms = np.linalg.norm(first, axis=1)
    if not np.allclose(norms, 1.0, atol=NORM_ATOL, rtol=0.0):
        raise RuntimeError("PREFLIGHT vectors are not normalized")
    cosine = np.sum(first * second, axis=1)
    print({
        "status": "RUNTIME_PREFLIGHT_ONLY",
        "runner": RUNNER,
        "caption_ids": [key for key, _text in selected],
        "caption_videos": selected_source,
        "shape": list(first.shape),
        "dtype": str(first.dtype),
        "max_abs_diff": float(np.max(np.abs(first - second))),
        "repeat_cosine": cosine.tolist(),
        "checkpoint": CHECKPOINT,
        "resolved_revision": MODEL_REVISION,
        "model_snapshot_fingerprint": MODEL_SNAPSHOT_FINGERPRINT,
        "packages": PACKAGE_VERSIONS,
        "writes_to_output_root": False,
        "numeric_parity_claimed": False,
    })
    unload_models()
else:
    print("PREFLIGHT skipped because RUN_SCOPE=SHARD")

## 6 — SHARD writer, resume contract và per-video processing

In [ ]:
class QwenCaptionLayout:
    """Narrow read-only adapter used by stage_n6_bge_m3."""

    def part(self, artifact: str, video_id: str) -> str:
        if artifact != "qwen_captions":
            raise RuntimeError(f"Notebook 28 may only read qwen_captions, got {artifact}")
        return f"{QWEN_ROOT}/parts/qwen_captions/{video_id}.json"


class ResearchBgeWriter:
    """Write one research NPZ without touching BundleLayout markers."""

    def __init__(self, transport: Any, video_id: str, expected_ids: list[str], workdir: Path):
        self.transport = transport
        self.video_id = video_id
        self.expected_ids = expected_ids
        self.workdir = workdir
        self.artifact: dict[str, Any] | None = None

    def write_vector_part(
        self,
        artifact: str,
        video_id: str,
        ids: list[str],
        vectors: np.ndarray,
        *,
        model: str,
        params: dict | None = None,
        elapsed_s: float | None = None,
    ) -> dict[str, Any]:
        if artifact != "dense_vectors" or video_id != self.video_id or ids != self.expected_ids:
            raise RuntimeError(f"{video_id}: stage output identity/artifact drift")
        checked = check_vectors(ids, vectors, video_id, VECTOR_DIM)
        if checked.dropped or checked.kept_ids != ids:
            raise RuntimeError(f"{video_id}: BGE stage dropped or reordered rows")
        local = self.workdir / "parts" / "dense_vectors" / f"{video_id}.npz"
        local.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(local, ids=np.asarray(ids, dtype=np.str_), vectors=checked.kept.astype(np.float32))
        audit = inspect_bge_npz(local, expected_ids=ids, video_id=video_id)
        remote = f"{OUTPUT_ROOT}/parts/dense_vectors/{video_id}.npz"
        digest = sha256_file(local)
        self.transport.put(local, remote)
        if self.transport.sha256(remote) != digest:
            raise RuntimeError(f"{video_id}: BGE NPZ upload hash mismatch")
        self.artifact = {
            "remote": remote,
            "sha256": digest,
            "rows": audit["rows"],
            "bytes": local.stat().st_size,
            "dim": audit["dim"],
            "dtype": audit["dtype"],
            "max_norm_error": audit["max_norm_error"],
            "model": model,
            "params": params or {},
            "elapsed_s": elapsed_s,
        }
        return {"artifact": artifact, **self.artifact}


def upload_json_readback(payload: dict[str, Any], local: Path, remote: str, *, hash_field: str) -> str:
    signed = dict(payload)
    signed[hash_field] = sha256_json(signed)
    atomic_write_json(local, signed)
    local_sha = sha256_file(local)
    research.put(local, remote)
    if research.sha256(remote) != local_sha:
        raise RuntimeError(f"JSON upload hash mismatch: {remote}")
    readback = REMOTE / "readback" / local.name
    research.get(remote, readback)
    if sha256_file(readback) != local_sha:
        raise RuntimeError(f"JSON read-back hash mismatch: {remote}")
    return local_sha


def expected_resume_fields(
    video_id: str,
    rows: list[dict[str, Any]],
    captions: dict[str, str],
    provenance: dict[str, Any],
) -> dict[str, Any]:
    return {
        "schema_version": 1,
        "status": "COMPLETE",
        "pipeline": PIPELINE,
        "runner": RUNNER,
        "video_id": video_id,
        "assigned_frame_count": len(rows),
        "caption_row_count": len(captions),
        "vector_row_count": len(captions),
        "keyframe_ids_sha256": sha256_json([str(row["keyframe_id"]) for row in rows]),
        "qwen_marker_payload_hash": provenance["qwen_marker_payload_hash"],
        "qwen_caption_sha256": provenance["qwen_caption_sha256"],
        "checkpoint": CHECKPOINT,
        "model_revision": MODEL_REVISION,
        "model_snapshot_fingerprint": MODEL_SNAPSHOT_FINGERPRINT,
        "environment": PACKAGE_VERSIONS,
        "vector_dim": VECTOR_DIM,
        "vector_dtype": VECTOR_DTYPE,
        "normalization": {"type": "l2", "atol": NORM_ATOL},
        "parity_freeze_sha256": EXPECTED_PARITY_FREEZE_SHA256,
        "nb24_complete_sha256": EXPECTED_NB24_COMPLETE_SHA256,
        "heldout_accessed": False,
        "btc_mutated": False,
    }


if RUN_SCOPE == "SHARD":
    validate_run_lock(RUNNER, RUNNERS, RUN_SCOPE, CONFIRM_SHARD)
    # Preload the exact resolved snapshot once. stage_n6_bge_m3 uses the same
    # cache key, so every video reuses this pinned instance instead of resolving
    # the model name again or loading 688 times.
    load_bge_model(lambda: SentenceTransformer(str(snapshot_dir)), cached_model)
    started = time.perf_counter()
    failures: list[dict[str, str]] = []
    verified: dict[str, dict[str, Any]] = {}
    for ordinal, video_id in enumerate(sorted(assigned), 1):
        if ordinal > 1 and time.perf_counter() - started >= SESSION_MAX_MINUTES * 60:
            break
        rows = assigned[video_id]
        expected_assigned_ids = [str(row["keyframe_id"]) for row in rows]
        try:
            captions, provenance = load_and_validate_qwen_video(
                research,
                WORK,
                runner=RUNNER,
                video_id=video_id,
                expected_ids=expected_assigned_ids,
                expected_nb24_sha=EXPECTED_NB24_COMPLETE_SHA256,
                hash_json=sha256_json,
                hash_file=sha256_file,
            )
            caption_ids = sorted(captions)
            fields = expected_resume_fields(video_id, rows, captions, provenance)
            vector_remote = f"{OUTPUT_ROOT}/parts/dense_vectors/{video_id}.npz"
            marker_remote = f"{OUTPUT_ROOT}/markers/{RUNNER}/{video_id}.json"
            existing = completed(
                research,
                WORK,
                marker_remote=marker_remote,
                expected_marker_fields=fields,
                vector_remote=vector_remote,
                expected_ids=caption_ids,
                video_id=video_id,
                hash_json=sha256_json,
                hash_file=sha256_file,
            )
            if existing is not None:
                verified[video_id] = existing
                print({"resume_skip": video_id, "vectors": len(caption_ids)}, flush=True)
                continue

            print(f"[{ordinal}/{len(assigned)}] {video_id}: {len(caption_ids)}/{len(rows)} captions -> BGE", flush=True)
            writer = ResearchBgeWriter(research, video_id, caption_ids, WORK)
            ctx = StageContext(
                video_id,
                Path("/dev/null"),
                [],
                writer,
                QwenCaptionLayout(),
                WORK,
                params={"batch_size": BGE_BATCH_SIZE},
            )
            stage_marker = stage_n6_bge_m3(ctx)
            if writer.artifact is None or int(stage_marker.get("rows", -1)) != len(caption_ids):
                raise RuntimeError(f"{video_id}: BGE stage row count mismatch")
            marker = {
                **fields,
                "artifact": writer.artifact,
                "repo_commit": repo_commit,
                "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            marker_local = OUTPUT / "markers" / RUNNER / f"{video_id}.json"
            upload_json_readback(marker, marker_local, marker_remote, hash_field="marker_payload_hash")
            final = completed(
                research,
                WORK,
                marker_remote=marker_remote,
                expected_marker_fields=fields,
                vector_remote=vector_remote,
                expected_ids=caption_ids,
                video_id=video_id,
                hash_json=sha256_json,
                hash_file=sha256_file,
            )
            if final is None:
                raise RuntimeError(f"{video_id}: final BGE marker verification failed")
            verified[video_id] = final
        except Exception as error:
            failure = {"video_id": video_id, "type": type(error).__name__, "error": str(error)}
            failures.append(failure)
            print({"failed": failure}, flush=True)

    # Verify every assigned video from Modal again before closing the runner manifest.
    verified = {}
    for video_id in sorted(assigned):
        rows = assigned[video_id]
        expected_assigned_ids = [str(row["keyframe_id"]) for row in rows]
        try:
            captions, provenance = load_and_validate_qwen_video(
                research,
                WORK,
                runner=RUNNER,
                video_id=video_id,
                expected_ids=expected_assigned_ids,
                expected_nb24_sha=EXPECTED_NB24_COMPLETE_SHA256,
                hash_json=sha256_json,
                hash_file=sha256_file,
            )
            caption_ids = sorted(captions)
            fields = expected_resume_fields(video_id, rows, captions, provenance)
            marker = completed(
                research,
                WORK,
                marker_remote=f"{OUTPUT_ROOT}/markers/{RUNNER}/{video_id}.json",
                expected_marker_fields=fields,
                vector_remote=f"{OUTPUT_ROOT}/parts/dense_vectors/{video_id}.npz",
                expected_ids=caption_ids,
                video_id=video_id,
                hash_json=sha256_json,
                hash_file=sha256_file,
            )
            if marker is not None:
                verified[video_id] = marker
        except Exception as error:
            failures.append({"video_id": video_id, "type": type(error).__name__, "error": str(error)})

    manifest = build_runner_manifest(
        runner=RUNNER,
        assigned=assigned,
        verified=verified,
        failures=failures,
        environment=PACKAGE_VERSIONS,
        repo_commit=repo_commit,
        model_revision=MODEL_REVISION,
        model_fingerprint=MODEL_SNAPSHOT_FINGERPRINT,
    )
    manifest["created_at_utc"] = datetime.now(timezone.utc).isoformat()
    manifest_local = OUTPUT / "runner-manifests" / f"{RUNNER}.json"
    manifest_remote = f"{OUTPUT_ROOT}/runner-manifests/{RUNNER}.json"
    manifest_sha = upload_json_readback(
        manifest,
        manifest_local,
        manifest_remote,
        hash_field="manifest_payload_hash",
    )
    unload_models()
    print({
        "B1_BGE_M3_PRODUCER": manifest["status"],
        "runner": RUNNER,
        "verified_videos": manifest["verified_videos"],
        "assigned_videos": manifest["assigned_videos"],
        "vector_rows": manifest["vector_rows"],
        "assigned_frames": manifest["assigned_frames"],
        "caption_coverage": manifest["caption_coverage"],
        "manifest_sha256": manifest_sha,
        "full_feature_parity_claimed": False,
        "official_ab_claimed": False,
        "production_go_claimed": False,
    })
else:
    print("SHARD not executed. Review PREFLIGHT, pin package versions, then set RUN_SCOPE='SHARD' and CONFIRM_SHARD=RUNNER.")

## 7 — Điều kiện chuyển từ PREFLIGHT sang SHARD

Chỉ đổi `RUN_SCOPE="SHARD"` và `CONFIRM_SHARD=RUNNER` khi:

1. PREFLIGHT in `RUNTIME_PREFLIGHT_ONLY` và shape `(2, 1024)`, `float32`, finite,
   L2-normalized.
2. Qwen runner manifest của đúng runner là `COMPLETE`, assigned/verified video
   và frame bằng nhau.
3. Bốn package version đã được pin/review để hai runner không tạo hai phép đo.
4. Resolved model revision/fingerprint đã được lưu lại.

Notebook 28 hoàn tất chỉ có nghĩa **BGE feature shard được sản xuất và xác minh**.
Nó không phải full feature parity, official B0/B1 A/B hay production GO. Aggregate,
snapshot/index và đánh giá end-to-end là các gate sau.